# Notebook 04: Interpretable "Glass-Box" Tree Models
## EBM (Explainable Boosting Machine), RuleFit & FIGS for Regulated Tabular Systems

---

### 1. Executive Intuition & The Interpretability Imperative

#### The Mental Model: Why Post-Hoc Explanations (SHAP) Are Not Enough
In high-stakes enterprise systems (Fintech lending, credit risk, healthcare diagnostics, and audit-compliant e-commerce billing), deploying a 1,000-tree XGBoost or LightGBM model introduces major legal and compliance hazards:
1. **The "Black-Box" Problem**: A prediction is the sum of thousands of split decisions. No human auditor can inspect the model and verify that it will never make a catastrophic or biased decision.
2. **The SHAP Limitation**: Post-hoc methods like SHAP or LIME are *local approximations*. They tell you what features were important *on average* for a specific sample, but they do not guarantee that the underlying model's behavior is monotonic or sensible across the entire feature domain.

**The Glass-Box Revolution**:
What if we can build a model that has **the exact same accuracy as LightGBM/XGBoost**, but is **100% mathematically transparent by design**?
This is achieved by **EBM (Explainable Boosting Machine)** from Microsoft Research, using Generalized Additive Models with pairwise interactions ($\text{GA}^2\text{M}$).

```mermaid
graph LR
    Input[Customer Transaction] --> Spline1[Univariate Shape f_1: order_value_sar]
    Input --> Spline2[Univariate Shape f_2: governorate_ar]
    Input --> Spline3[Univariate Shape f_3: payment_method_ar]
    Input --> Interaction[Bivariate Surface f_12: order_value x payment_method]
    Spline1 & Spline2 & Spline3 & Interaction --> Sum((Additive Sum: beta_0 + sum f_i))
    Sum --> Softmax[Softmax Link: Exact Probability for 20 Classes]
```

--- 

### 2. Deep Mathematical Derivations

#### A. Generalized Additive Models with Interactions ($\text{GA}^2\text{M}$)
Standard Linear Models assume simple independent linearity:
$$g(E[Y]) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_D x_D$$
Unconstrained GBDTs model arbitrary high-order interactions: $g(E[Y]) = F(x_1, x_2, \dots, x_D)$, which destroys human interpretability.

**EBM optimizes an exact $\text{GA}^2\text{M}$ formulation**:
$$g(E[Y]) = \beta_0 + \sum_{i=1}^{D} f_i(x_i) + \sum_{i < j} f_{ij}(x_i, x_j)$$
where:
* $\beta_0$ is the global intercept.
* $f_i(x_i)$ is an arbitrary non-linear univariate shape function for feature $i$.
* $f_{ij}(x_i, x_j)$ is a 2-dimensional interaction surface for feature pair $(i, j)$.
* $g(\cdot)$ is the link function (Softmax for our 20-class classification problem).

##### 1. Round-Robin Cyclic Gradient Boosting
How does EBM learn arbitrary non-linear functions $f_i(x_i)$ without allowing features to interact?
It uses **Cyclic Boosting**:
1. At iteration 1, fit a shallow tree stump strictly on feature $x_1$ to update $f_1(x_1)$.
2. At iteration 2, fit a shallow tree stump strictly on feature $x_2$ to update $f_2(x_2)$.
3. Continue in a round-robin cycle across all $D$ features: $x_1 \to x_2 \to \dots \to x_D \to x_1 \dots$
4. Use a very small learning rate (e.g. $\eta = 0.01$) over thousands of cycles.

**Mathematical Consequence**: Because each step is strictly 1-dimensional, feature $x_i$ can **never collinearize or confound** with feature $x_j$. The resulting shape function $f_i(x_i)$ represents the *pure marginal impact* of feature $i$ on the log-odds of each of the 20 departments!

##### 2. FAST (Fast Adaptive Screening of Interactions)
There are $\binom{D}{2} = \frac{D(D-1)}{2}$ possible pairs of features (for $D=200$, that is $19,900$ pairs!).
EBM evaluates all pairs using the FAST algorithm:
1. Compute the residual errors after all main effects $f_i$ have converged.
2. Approximate the interaction energy for each pair $(i, j)$ using tensor products.
3. Retain only the top $K$ interaction pairs (e.g. top 10 pairs), and add them as bivariate splines $f_{ij}(x_i, x_j)$.

#### B. RuleFit: Tree Branches as Sparse $L_1$ Linear Rules
Proposed by Jerome Friedman, RuleFit bridges the gap between tree ensembles and linear models:

##### Step 1: Decision Rule Extraction
Train a shallow tree ensemble (e.g. 50 trees of depth 3). Every path from root to leaf defines a conjunctive binary rule:
$$r_m(x) = \prod_{j \in \text{Path}_m} \mathbb{I}(s_{mj}^{\text{low}} < x_j \le s_{mj}^{\text{high}}) \in \{0, 1\}$$
> **Example E-Commerce Rule**:
> $r_1(x) = \mathbb{I}(\text{order\_value} > 500) \times \mathbb{I}(\text{payment\_method} == \text{'تمارا'}) \times \mathbb{I}(\text{device} == \text{'iOS App'})$

##### Step 2: Sparse Regularized Optimization (Lasso)
Extract all $M$ unique rules from the trees. Treat every rule $r_m(x)$ as a binary feature alongside the original linear features $x_j$, and optimize an ElasticNet / Lasso objective:
$$\min_{\beta_0, \beta, \alpha} \left[ \frac{1}{N} \sum_{i=1}^{N} L\left(y_i, \beta_0 + \sum_{j=1}^{D} \beta_j x_{ij} + \sum_{m=1}^{M} \alpha_m r_m(x_i)\right) + \lambda \sum_{m=1}^{M} |\alpha_m| + \lambda \sum_{j=1}^{D} |\beta_j| \right]$$

##### Step 3: Extreme Sparsity
Due to the geometry of the $L_1$ penalty $|\alpha_m|$, the optimizer drives 95% of rule coefficients $\alpha_m$ strictly to zero!
The surviving 10–20 rules constitute a transparent scoring policy that can be **directly exported into SQL database queries**:
```sql
CASE 
    WHEN order_value > 500 AND payment_method = 'تمارا' THEN +2.15
    WHEN session_seconds > 600 AND discount_pct > 0.3 THEN +1.80
    ELSE -0.45
END
```

In [ ]:
import sys, os
cur_dir = os.path.abspath(os.getcwd())
trees_dir = os.path.abspath(os.path.join(cur_dir, '..')) if os.path.basename(cur_dir) == 'notebooks' else cur_dir
repo_root = os.path.abspath(os.path.join(trees_dir, '..'))
for p in [trees_dir, repo_root]:
    if p not in sys.path: sys.path.insert(0, p)
# Setup environment
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if root_dir not in sys.path:

from trees.utils import (
    print_hardware_summary, load_dataset, prepare_features,
    evaluate_multiclass_model, plot_confusion_matrix_20, plot_metrics_comparison,
    DEPARTMENTS_EN
)

print_hardware_summary()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Load 25,000 samples for glass-box training
df = load_dataset(sample_rows=25_000)
X_raw, y, cat_cols, num_cols = prepare_features(df)

# Preprocess categoricals
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = X_raw.copy()
X_encoded[cat_cols] = encoder.fit_transform(X_raw[cat_cols])

# 70/10/20 Stratified Split
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.6667, random_state=42, stratify=y_temp)

print(f"Train instances: {len(X_train):,} | Test instances: {len(X_test):,}")

### 3. Model 1: Explainable Boosting Classifier (EBM / InterpretML)
We train Microsoft's EBM, demonstrating exact non-linear additive shape functions for our 20-class dataset.

In [ ]:
from interpret.glassbox import ExplainableBoostingClassifier

all_results = []

t0 = time.time()
ebm = ExplainableBoostingClassifier(
    max_bins=128,
    interactions=10,
    outer_bags=8,
    inner_bags=0,
    n_jobs=-1,
    random_state=42
)
ebm.fit(X_train, y_train)
ebm_time = time.time() - t0

ebm_pred = ebm.predict(X_test)
ebm_prob = ebm.predict_proba(X_test)
ebm_metrics = evaluate_multiclass_model("Explainable Boosting Machine (EBM)", y_test, ebm_pred, ebm_prob, ebm_time)
all_results.append(ebm_metrics)
print("EBM Evaluation:", ebm_metrics)

### 4. Inspecting Glass-Box Feature Shapes
We extract and visualize global feature scores and interaction terms.

In [ ]:
ebm_global = ebm.explain_global(name="EBM Explanations")
global_data = ebm_global.data()
feat_names = global_data['names'][:15]
feat_scores = global_data['scores'][:15]

plt.figure(figsize=(10, 5))
plt.barh(feat_names, feat_scores, color='darkorchid', edgecolor='black', alpha=0.8)
plt.title('Top 15 Global Feature Importances (EBM Glass-Box)', fontsize=12, fontweight='bold')
plt.xlabel('Mean Absolute Contribution Score', fontsize=11)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 5. Model 2: FIGS (Fast Interpretable Greedy-Tree Sums)
Evaluating an additive ensemble of very shallow concurrent trees.

In [ ]:
try:
    from imodels import FIGSClassifier
    
    t0 = time.time()
    figs = FIGSClassifier(max_rules=20, random_state=42)
    figs.fit(X_train, y_train)
    figs_time = time.time() - t0
    
    figs_pred = figs.predict(X_test)
    figs_prob = figs.predict_proba(X_test)
    figs_metrics = evaluate_multiclass_model("FIGS (imodels)", y_test, figs_pred, figs_prob, figs_time)
    all_results.append(figs_metrics)
except ImportError:
    print("imodels not installed; skipping FIGS.")

pd.DataFrame(all_results)[['Model', 'Training Time (s)', 'Top-1 Accuracy', 'Top-3 Accuracy', 'Macro F1-Score']]

### 6. Architectural Decision Matrix: Interpretability vs. Accuracy

| Criterion | Standard GBDT (XGBoost/LightGBM) | EBM (Explainable Boosting Machine) | Single Decision Tree |
|---|---|---|---|
| **Accuracy** | SOTA (Highest) | Competitive with GBDT | Low |
| **Interpretability** | Black-Box (Requires SHAP) | **100% Glass-Box (Exact Math)** | Transparent but brittle |
| **Feature Attributions**| Approximations (Sampling noise)| **Exact Marginal Shapes** | Split path inspection |
| **Regulatory Compliance**| Hard to audit (GDPR risk) | **Compliant by design** | Compliant |
| **Training Speed** | Fast (seconds on GPU) | Moderate (round-robin cyclic) | Ultra-fast |